# Demo **Paraphrasing**

Our imposter approach needs models to reword an original text keeping the same meaning.
Ideally, we want to model the way texts originated.
This is dependent on the text's genre, i.e. a student essay originates from a task description and a list of URLs while a new article originates from a list of additional sources or from spectating an event.
We implemented both a naive approach to rewording a text and a more sophisticated one:
- given a prompt and a text, the naive approach uses a language model to paraphrase the text
- given a prompt and a text LLM A is used to extract the text's genre, tone and summary of bullet points, then LLM B is used to reword the text in the same genre and tone based on the summary of bullet points

In [28]:
from pathlib import Path
import os
import textwrap
import sys

# Add the root of the project to Python path
sys.path.append(os.path.abspath(".."))
from genai_detection.config import CONFIG
from genai_detection.paraphrasing.paraphraser import (
    T5ChatGPTParaphraser,
    T5GooglePAWSParaphraser,
    TranslationParaphraser,
    BulletPointParaphraser,
    # OllamaParaphraser, token expired
    TaskParaphraser,
    TopicParaphraser,
    TitleParaphraser,
    SAIAParaphraser,
)

## Original text
We use a CNN news article from 23.06.2025/ 04.07.2025 as an example.

In [29]:
path2datasets = (
    Path(os.getcwd()).resolve().parent / "data" / "datasets" / "custom_texts"
)
data_category = "News"  # This is used for plot titles in the notebook
path2results = Path(os.getcwd()).resolve().parent / CONFIG.SAVE_PATH / "paraphrasing"
assert path2datasets.exists(), f"Path to datasets {path2datasets} does not exist."
# file_name = "cnn_230625"    # USA attacks Iran
file_name = "cnn_040725"  # Dalai Lama
original_text = open(path2datasets / f"{file_name}.txt").read()

print(textwrap.fill(original_text, width=80))

  For much of the past century, the Dalai Lama has been the living embodiment of
Tibet’s struggle for greater freedoms under Chinese Communist Party rule,
sustaining the cause from exile even as an increasingly powerful Beijing has
become ever more assertive in suppressing it.  As his 90th birthday approaches
this Sunday, the spiritual leader for millions of followers of Tibetan Buddhism
worldwide is bracing for a final showdown with Beijing: the battle over who will
control his reincarnation.  On Wednesday, the Dalai Lama announced that he will
have a successor after his death, and that his office will have the sole
authority to identify his reincarnation.  “I am affirming that the institution
of the Dalai Lama will continue,” the Nobel Peace laureate said in a video
message to religious elders gathering in Dharamshala, India, where he has found
refuge since Chinese communist troops put down an armed uprising in his
mountainous homeland in 1959.  The cycle of rebirth lies at the core 

## Naive Approach

In [30]:
paraphrasers = {
    "T5_ChatGPT": T5ChatGPTParaphraser(),
    "T5_Google_PAWS": T5GooglePAWSParaphraser(),
    "SAIA": SAIAParaphraser(model_id=CONFIG.SAIA_MODEL),
}

Using T5ChatGPTParaphraser
Loaded T5ChatGPTParaphraser tokenizer
Loaded T5ChatGPTParaphraser model
Using T5GooglePAWSParaphraser
Loaded T5GooglePAWSParaphraser tokenizer
Loaded T5GooglePAWSParaphraser model


In [31]:
n_responses = 1  # number of paraphrases to generate
max_length = 512  # Maximum length of the generated paraphrase
temperature = 0.7  # Controls the randomness of the output. Lower values make the output more deterministic.

prompts = [
    "Paraphrase the text above without changing its meaning. Use different words and vary the sentence structure while maintaining a consistent tone. Your paraphrase should be three times as long than the original. Output only the paraphrased sentence, with NO explanations or extra text.",
    "Paraphrase the given sentence by identifying the main subject, verb, and object. Replace each with synonyms or closely related words, adjusting grammar naturally. Keep the new sentence close in length to the original. Output only the final paraphrased sentence.",
    "Paraphrase the sentence above without changing its meaning. Use different words and vary the sentence structure while keeping the tone consistent. Keep the new sentence similar in length to the original. Output only the paraphrased sentence, with no explanations or extra text.",
]

In [32]:
for paraphraser_name, paraphraser in paraphrasers.items():
    print(f"\n{'='*40}")
    print(f"\nUsing paraphraser: {paraphraser_name}")
    for prompt in prompts:
        print(f"\nPrompt: {prompt}")
        paraphrases = paraphraser.paraphrase(
            original_text,
            n_responses=n_responses,
            max_length=max_length,
            temperature=temperature,
            prompt=prompt,
        )
        for i, paraphrase in enumerate(paraphrases):
            print(f"Paraphrase {i+1}: {textwrap.fill(paraphrase, width=80)}")



Using paraphraser: T5_ChatGPT

Prompt: Paraphrase the text above without changing its meaning. Use different words and vary the sentence structure while maintaining a consistent tone. Your paraphrase should be three times as long than the original. Output only the paraphrased sentence, with NO explanations or extra text.
T5 model
Paraphrase 1: Tibetan Buddhists are bracing themselves for a potential confrontation with
Beijing, who has been resisting the Chinese Communist Party's efforts to bring
about their own political and economic reforms. The Dalai Lama's death could
lead to a confrontation with Beijing, who claims to have no authority over the
group.

Prompt: Paraphrase the given sentence by identifying the main subject, verb, and object. Replace each with synonyms or closely related words, adjusting grammar naturally. Keep the new sentence close in length to the original. Output only the final paraphrased sentence.
T5 model
Paraphrase 1: Tibetan Buddhists are bracing themselves

## Bullet Point Approach

Ollama models are by far the best ones for extracting the genre, tone and summary of bullet points and returning them in a structured way.
Hence, we use the Ollama model as the text extractor and any other model as the text generator.

In [33]:
models = {
    "text_extractor": {
        "name": "GPT-oss-120B",
        "model": SAIAParaphraser(model_id=CONFIG.SAIA_MODEL),
    },
    "text_generator": {
        "name": "GPT-oss-120B",
        "model": SAIAParaphraser(model_id=CONFIG.SAIA_MODEL),
    },
}
bullet_point_paraphraser = BulletPointParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = bullet_point_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")
# save to file
# with open(path2datasets / f"paraphrase_{file_name}_e={models["text_extractor"]['name']}_g={models["text_generator"]['name']}_{i+1}.txt", "w") as f:
#     f.write(paraphrase)

Paraphrase 1:
The world’s most widely recognised Buddhist leader has announced that he will turn 90 years old this year, a milestone that is being marked with a series of statements that could reshape the future of Tibetan Buddhism and the political landscape of the Himalayan region. In a televised address delivered from his residence in Dharamshala, the 90‑year‑old spiritual head declared that, following his death, the responsibility for identifying his reincarnation will rest exclusively with his own office. He added that the child who will be recognized as the next Dalai Lama is expected to be born outside the borders of the People’s Republic of China, a pronouncement that directly challenges the claim made by the Chinese Communist Party that it alone holds the authority to approve any future Dalai Lama and other senior lamas.

The Dalai Lama’s statement was accompanied by a firm affirmation that the institution he leads will continue to function after his passing. “The spiritual li

## Task Paraphraser
The task paraphraser extracts the task, tone and genre from the text and then generates a paraphrase based on this information.

In [34]:
task_paraphraser = TaskParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = task_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Paraphrase 1:
The world will mark a rare milestone on March 6, 2025, as the 90th birthday of His Holiness the 14th Dalai Lama draws near. The Tibetan spiritual leader, who has spent more than six decades in exile and become an emblem of non‑violent resistance, will celebrate the occasion with a series of public events in Dharamshala, India, and a televised address to a global audience. Yet the celebrations are set against a backdrop of renewed political tension, as the Dalai Lama’s office announced last week that it will assume responsibility for selecting his next incarnation—a decision that reverberates far beyond the confines of Tibetan Buddhism.

The declaration came in a brief statement released by the Office of His Holiness, which said: “In accordance with the traditions of our faith and the guidance of senior lamas, the Office of His Holiness will oversee the identification of the next incarnation of the Dalai Lama.” The wording, deliberately vague, sidesteps the long‑standing d

## Topic Paraphraser
The topic paraphraser extracts the topic, tone and genre from the text and then generates a paraphrase based on this information.

In [35]:
topic_paraphraser = TopicParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = topic_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Topic model
Paraphrase 1:
The looming succession of the 15th Dalai Lama has become a flashpoint in the already volatile China‑Tibet relationship, exposing a clash between religious tradition, state sovereignty, and international human‑rights norms.

Since the exile of the 14th Dalai Lama in 1959, Beijing has pursued a policy of controlling the identification of high‑level reincarnations, arguing that such matters fall under its constitutional jurisdiction. The Chinese government formalised this stance in 2007 with the State Religious Affairs Bureau’s “Regulations on the Management of Religious Affairs”, which explicitly require all reincarnation processes to receive state approval. Tibetan Buddhists, however, maintain that the recognition of a Dalai Lama follows centuries‑old spiritual criteria, including visions, tests, and the guidance of senior lamas, a procedure the exile community insists must remain free from political interference.

In early 2025, the Central Tibetan Administrat

## Title Paraphraser
The title paraphraser extracts the title, tone and genre from the text and then generates a paraphrase based on this information.

In [36]:
title_paraphraser = TitleParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = title_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Title model
Paraphrase 1:
**Dalai Lama’s Succession Battle: Tibet, China, and the Future of Reincarnation**

The death of the 14th Dalai Lama in 2023 has revived a contest that has simmered for decades, pitting the exiled Tibetan community against the People’s Republic of China over the very mechanism that legitimizes spiritual and political authority in Tibetan Buddhism. While the Dalai Lama’s own statements have emphasized a secular, democratic future for Tibet, the question of who will be recognized as his reincarnation has become a proxy for broader geopolitical struggles, cultural survival, and the evolving role of reincarnation in a modern world.

In the weeks following the Dalai Lama’s passing, the Central Tibetan Administration (CTA) in Dharamshala announced the formation of a committee of senior monks tasked with locating the next incarnation according to traditional methods. The committee’s charter explicitly rejects any involvement of the Chinese government, citing the 2008 

# Translation Paraphraser

In [37]:
translation_paraphraser = TranslationParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
    language="French",
)
paraphrases = translation_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

[DEBUG] Using TranslationParaphraser with prompt: Translate the text above into French. Do not use direct quotes or newlines. Output only the translated text, without any additional commentary or formatting.
Paraphrase 1:


For most of the last century, the Dalai Lama has been the living embodiment of Tibet's struggle for greater freedoms under the domination of the Chinese Communist Party, supporting the cause from exile even as the increasingly powerful Beijing asserted its repressive stance.

As he approaches his 90th birthday on Sunday, the spiritual leader of millions of followers of Tibetan Buddhism worldwide is preparing for a final showdown with Beijing: the battle over who will control his reincarnation.

On Wednesday, the Dalai Lama announced that he would have a successor after his death, and that only his office would be able to identify his reincarnation.

"I affirm that the institution of the Dalai Lama will continue," said the Nobel Peace Prize laureate in a video messag

## Batch Processing

In [38]:
# Batch paraphrasing
texts = [
    "The sky is blue.",
    "I love programming.",
    "Artificial intelligence is fascinating.",
]
paraphrased_batch = bullet_point_paraphraser.paraphrase_batch(texts)
for original, paraphrased in zip(texts, paraphrased_batch):
    print(f"Original: {original} | Paraphrased: {paraphrased}")

Original: The sky is blue. | Paraphrased: ['Azure Daylight Sky Phenomenon', 'Azure sky: scattering, culture', 'Azure daylight sky, worldwide', 'Azure sky scattering sunlight', 'Blue daylight scattering worldwide']
Original: I love programming. | Paraphrased: ['Coding rocks me!', 'Coding fuels me!', 'Coding fuels me!', 'I love coding!', 'Coding thrills me!']
Original: Artificial intelligence is fascinating. | Paraphrased: ['AI transforms, learns, discovers, challenges.', 'AI fuels change, responsibly!', 'AI revolutionizes everything responsibly!', 'AI transforms, learns, discovers, evolves!', 'AI transforms everything, responsibly!']
